## EXPERIMENT_NAME = "PHASE2_STRICT_K6"

What is Testing in Phase 2

Testing:

Does increasing temporal coverage improve true generalization?

NOT testing leakage again.

So Phase 2 should contain:

Strict split only

No leaky code

Clean evaluation

In [1]:

# =========================
# 0) Imports + safety
# =========================
import os, math, random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import models, transforms

from PIL import Image, ImageSequence, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True  # prevents crashes on truncated GIFs

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

In [2]:

# =========================
# 1) Config (match Zhang Table 1)
# =========================
CSV_PATH = r"D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\gifgif_emotion_labels.csv"   # <-- CHANGE
GIF_ROOT = None  # set to folder if CSV has relative paths, else keep None

Z_CLASSES = ["happiness","satisfaction","surprise","sadness","anger","disgust","fear"]

SEQ_LEN = 4
TARGET_SIZE = (224,224)
MAX_FRAMES_PER_GIF = 300

BATCH = 30
LR = 1e-4
WD = 1e-4
EPOCHS = 30

FREEZE_EPOCHS = 0  # Phase 1 reproduction: keep 0 (Zhang does not mention freezing)


In [3]:

# =========================
# 2) Load CSV + build Zhang-7 label from score columns (argmax over 7 scores)
# =========================
df = pd.read_csv(CSV_PATH)

if GIF_ROOT is not None:
    def _make_abs(p):
        if isinstance(p,str) and (os.path.isabs(p) or (len(p)>2 and p[1]==":")):
            return p
        return os.path.join(GIF_ROOT, str(p))
    df["gif_path"] = df["gif_path"].apply(_make_abs)

score_cols = [f"{c}_score" for c in Z_CLASSES]
missing = [c for c in score_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing score columns: {missing}")

df[score_cols] = df[score_cols].apply(pd.to_numeric, errors="coerce")
print("NaNs per score col:\n", df[score_cols].isna().sum())

df["label"] = df[score_cols].idxmax(axis=1).str.replace("_score","", regex=False)
df7 = df[df["label"].isin(Z_CLASSES)].copy()

print("\nAvailable counts (7-class):")
print(df7["label"].value_counts())
print("Total 7-class rows:", len(df7))


NaNs per score col:
 happiness_score       0
satisfaction_score    0
surprise_score        0
sadness_score         0
anger_score           0
disgust_score         0
fear_score            0
dtype: int64

Available counts (7-class):
label
happiness       2000
sadness          998
anger            911
surprise         679
satisfaction     652
disgust          459
fear             402
Name: count, dtype: int64
Total 7-class rows: 6101


In [4]:

# =========================
# 3) Zhang subset: first 300 per class => 2100 GIFs
# =========================
parts = []
for lab in Z_CLASSES:
    sub = df7[df7["label"] == lab]
    if len(sub) < 300:
        raise ValueError(f"Not enough samples for {lab}: {len(sub)} < 300")
    parts.append(sub.iloc[:300].copy())

df2100 = pd.concat(parts, ignore_index=True)

print("Total selected:", len(df2100))
print(df2100["label"].value_counts())


Total selected: 2100
label
happiness       300
satisfaction    300
surprise        300
sadness         300
anger           300
disgust         300
fear            300
Name: count, dtype: int64


In [5]:

# =========================
# 4) STRICT split: split by GIF (80/20 per class), then chunk
# =========================
train_parts, test_parts = [], []
for lab in Z_CLASSES:
    sub = df2100[df2100["label"] == lab]
    tr, te = train_test_split(sub, test_size=0.2, random_state=SEED, shuffle=True)
    train_parts.append(tr); test_parts.append(te)

train_df = pd.concat(train_parts, ignore_index=True)
test_df  = pd.concat(test_parts,  ignore_index=True)

print("Train:", len(train_df), "Test:", len(test_df))
print(train_df["label"].value_counts())


Train: 1680 Test: 420
label
happiness       240
satisfaction    240
surprise        240
sadness         240
anger           240
disgust         240
fear            240
Name: count, dtype: int64


In [6]:

# =========================
# 5) Transforms (paper-faithful: no augmentation)
# =========================
base_tf = transforms.Compose([
    transforms.Resize(TARGET_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])
train_tf = base_tf
test_tf  = base_tf

label2id = {lab:i for i,lab in enumerate(Z_CLASSES)}
id2label = {i:lab for lab,i in label2id.items()}


In [7]:

# =========================
# 6) GIF helpers + chunking
# =========================
def load_gif_frames_full(path, max_frames=300):
    frames = []
    im = Image.open(path)
    for i, fr in enumerate(ImageSequence.Iterator(im)):
        if max_frames is not None and i >= max_frames:
            break
        frames.append(fr.convert("RGB"))
    return frames

def chunk_consecutive(frames, seq_len=4):
    if len(frames) == 0:
        return [[Image.new("RGB", TARGET_SIZE, (0,0,0)) for _ in range(seq_len)]]
    chunks = []
    # non-overlapping consecutive chunks (matches the paper phrasing)
    for start in range(0, len(frames), seq_len):
        clip = frames[start:start+seq_len]
        if len(clip) < seq_len:
            # pad by repeating last frame
            clip = clip + [clip[-1]] * (seq_len - len(clip))
        chunks.append(clip)
    return chunks


In [8]:
K_CLIPS_PER_GIF = 6  # Phase 2 setting

class GifChunkDataset(Dataset):
    def __init__(self, df, transform, seq_len=4, max_frames=300):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.seq_len = seq_len
        self.max_frames = max_frames

        self.bad_gifs = 0
        self.index = []  # list of (row_idx, chunk_idx)

        for i in range(len(self.df)):
            path = self.df.loc[i, "gif_path"]
            try:
                frames = load_gif_frames_full(path, max_frames=self.max_frames)
                chunks = chunk_consecutive(frames, seq_len=self.seq_len)

                if len(chunks) <= K_CLIPS_PER_GIF:
                    chosen = list(range(len(chunks)))
                else:
                    chosen = np.linspace(0, len(chunks) - 1, K_CLIPS_PER_GIF).round().astype(int).tolist()

                for c in chosen:
                    self.index.append((i, c))

            except Exception:
                self.bad_gifs += 1
                continue

        print(f"Built dataset: GIFs={len(self.df)}  Chunks={len(self.index)}  BadGIFs={self.bad_gifs}")

    def __len__(self):
        return len(self.index)

    def __getitem__(self, k):
        i, cidx = self.index[k]
        r = self.df.loc[i]
        lab = r["label"]
        path = r["gif_path"]

        try:
            frames = load_gif_frames_full(path, max_frames=self.max_frames)
            chunks = chunk_consecutive(frames, seq_len=self.seq_len)
            clip = chunks[min(cidx, len(chunks)-1)]
            x = torch.stack([self.transform(fr) for fr in clip], dim=0)  # [T,3,H,W]
        except Exception:
            x = torch.zeros((self.seq_len, 3, *TARGET_SIZE), dtype=torch.float32)

        y = label2id[lab]
        gif_id = r["gif_id"] if "gif_id" in self.df.columns else os.path.basename(path)
        return x, y, gif_id


In [9]:

# =========================
# 8) Build STRICT datasets + loaders (GIF-split -> chunk)
# =========================
train_ds = GifChunkDataset(train_df, transform=train_tf, seq_len=SEQ_LEN, max_frames=MAX_FRAMES_PER_GIF)
test_ds  = GifChunkDataset(test_df,  transform=test_tf,  seq_len=SEQ_LEN, max_frames=MAX_FRAMES_PER_GIF)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0, pin_memory=(device.type=="cuda"))
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=(device.type=="cuda"))

print("STRICT sanity:")
print("Train chunks:", len(train_ds), "Test chunks:", len(test_ds))
print("Bad train GIFs:", train_ds.bad_gifs, "Bad test GIFs:", test_ds.bad_gifs)

xb, yb, gid = next(iter(train_loader))
print("Batch:", xb.shape, yb.shape)


Built dataset: GIFs=1680  Chunks=7910  BadGIFs=0
Built dataset: GIFs=420  Chunks=1979  BadGIFs=0
STRICT sanity:
Train chunks: 7910 Test chunks: 1979
Bad train GIFs: 0 Bad test GIFs: 0
Batch: torch.Size([30, 4, 3, 224, 224]) torch.Size([30])


In [10]:

# =========================
# 9) Model: ResNet18 + ConvGRU (close to Zhang)
# =========================
class ConvGRUCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size=3):
        super().__init__()
        padding = kernel_size // 2
        self.hidden_dim = hidden_dim
        self.conv_z = nn.Conv2d(input_dim + hidden_dim, hidden_dim, kernel_size, padding=padding)
        self.conv_r = nn.Conv2d(input_dim + hidden_dim, hidden_dim, kernel_size, padding=padding)
        self.conv_h = nn.Conv2d(input_dim + hidden_dim, hidden_dim, kernel_size, padding=padding)

    def forward(self, x, h_prev):
        if h_prev is None:
            h_prev = torch.zeros(x.size(0), self.hidden_dim, x.size(2), x.size(3), device=x.device)
        cat = torch.cat([x, h_prev], dim=1)
        z = torch.sigmoid(self.conv_z(cat))
        r = torch.sigmoid(self.conv_r(cat))
        cat_r = torch.cat([x, r * h_prev], dim=1)
        h_hat = torch.tanh(self.conv_h(cat_r))
        h = (1 - z) * h_hat + z * h_prev
        return h

class ConvGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size=3):
        super().__init__()
        self.cell = ConvGRUCell(input_dim, hidden_dim, kernel_size)

    def forward(self, x_seq):
        h = None
        for t in range(x_seq.size(1)):
            h = self.cell(x_seq[:, t], h)
        return h

class ResNetConvGRU(nn.Module):
    def __init__(self, num_classes=7, gru_hidden=256, dropout=0.4):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(base.children())[:-2])  # [B,512,7,7]
        self.convgru = ConvGRU(input_dim=512, hidden_dim=gru_hidden, kernel_size=3)
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.drop = nn.Dropout(p=dropout)
        self.fc = nn.Linear(gru_hidden, num_classes)

    def forward(self, x):
        feats = []
        for t in range(x.size(1)):
            feats.append(self.backbone(x[:, t]))
        feats = torch.stack(feats, dim=1)      # [B,T,512,7,7]
        h = self.convgru(feats)                # [B,H,7,7]
        v = self.pool(h).flatten(1)
        v = self.drop(v)
        return self.fc(v)

def init_model():
    m = ResNetConvGRU(num_classes=len(Z_CLASSES), gru_hidden=256, dropout=0.4).to(device)
    return m

model = init_model()
print("Params (M):", sum(p.numel() for p in model.parameters())/1e6)


Params (M): 16.487495


In [12]:

# =========================
# 10) Optimizer + train/eval helpers
# =========================
def make_optimizer(model):
    return torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)

def run_epoch(model, loader, opt=None):
    train = opt is not None
    model.train(train)
    losses = []
    y_true, y_pred = [], []

    for x, y, _gid in tqdm(loader, leave=False):
        x, y = x.to(device), y.to(device)

        if train:
            opt.zero_grad(set_to_none=True)

        logits = model(x)
        loss = F.cross_entropy(logits, y)

        if train:
            loss.backward()
            opt.step()

        losses.append(loss.item())
        y_true.extend(y.detach().cpu().numpy().tolist())
        y_pred.extend(logits.argmax(1).detach().cpu().numpy().tolist())

    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return float(np.mean(losses)), float(acc), float(p), float(r), float(f1)


In [13]:

# =========================
# 11) TRAIN – STRICT (no leakage)
# =========================
model = init_model()
opt = make_optimizer(model)

history = []
CKPT_STRICT = "phase2_strict_k6_best.pt"

PATIENCE = 5
best_acc = -1.0
no_improve = 0

print("EPOCHS:", EPOCHS, "PATIENCE:", PATIENCE)
print("SEQ_LEN:", SEQ_LEN, "K_CLIPS_PER_GIF:", K_CLIPS_PER_GIF, "MAX_FRAMES_PER_GIF:", MAX_FRAMES_PER_GIF)
print("Train GIFs:", len(train_df), "Test GIFs:", len(test_df))
print("Train chunks:", len(train_ds), "Test chunks:", len(test_ds))


for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc, tr_p, tr_r, tr_f1 = run_epoch(model, train_loader, opt=opt)
    te_loss, te_acc, te_p, te_r, te_f1 = run_epoch(model, test_loader,  opt=None)

    row = dict(
        epoch=epoch,
        train_loss=tr_loss, train_acc=tr_acc, train_p=tr_p, train_r=tr_r, train_f1=tr_f1,
        test_loss=te_loss,  test_acc=te_acc,  test_p=te_p,  test_r=te_r,  test_f1=te_f1,
        bad_train_gifs=train_ds.bad_gifs, bad_test_gifs=test_ds.bad_gifs,
        train_chunks=len(train_ds), test_chunks=len(test_ds),
    )
    history.append(row)
    print(row)

    improved = te_acc > best_acc + 1e-4

    if improved:
        best_acc = te_acc
        no_improve = 0
        torch.save({"model": model.state_dict(), "epoch": epoch, "metrics": row}, CKPT_STRICT)
        print("   saved best ACC:", best_acc)
    else:
        no_improve += 1

    if no_improve >= PATIENCE:
        print(f" Early stopping: no improvement for {PATIENCE} epochs.")
        break

hist_df = pd.DataFrame(history)
hist_df.tail()

hist_df.to_csv("phase2_strict_k6_history.csv", index=False)
print("Saved history to phase2_strict_k6_history.csv")



EPOCHS: 30 PATIENCE: 5
SEQ_LEN: 4 K_CLIPS_PER_GIF: 6 MAX_FRAMES_PER_GIF: 300
Train GIFs: 1680 Test GIFs: 420
Train chunks: 7910 Test chunks: 1979


  0%|          | 0/264 [00:00<?, ?it/s]

  0%|          | 0/66 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 1.3247896808792243, 'train_acc': 0.5093552465233881, 'train_p': 0.5088150491117062, 'train_r': 0.508263642505451, 'train_f1': 0.5065730054814881, 'test_loss': 2.3176620476173633, 'test_acc': 0.2627589691763517, 'test_p': 0.28252310749697623, 'test_r': 0.26368015658271937, 'test_f1': 0.2612885817364744, 'bad_train_gifs': 0, 'bad_test_gifs': 0, 'train_chunks': 7910, 'test_chunks': 1979}
   saved best ACC: 0.2627589691763517


  0%|          | 0/264 [00:00<?, ?it/s]

...

  0%|          | 0/66 [00:00<?, ?it/s]

{'epoch': 5, 'train_loss': 0.07777870684036646, 'train_acc': 0.9756005056890013, 'train_p': 0.9755589051047842, 'train_r': 0.9755221820918487, 'train_f1': 0.9755363959669386, 'test_loss': 3.492495217106559, 'test_acc': 0.2425467407781708, 'test_p': 0.2491071643074804, 'test_r': 0.2404806929855454, 'test_f1': 0.23444230362826676, 'bad_train_gifs': 0, 'bad_test_gifs': 0, 'train_chunks': 7910, 'test_chunks': 1979}


  0%|          | 0/264 [00:00<?, ?it/s]

  0%|          | 0/66 [00:00<?, ?it/s]

{'epoch': 6, 'train_loss': 0.07988346603374477, 'train_acc': 0.9735777496839444, 'train_p': 0.9735576123465386, 'train_r': 0.9737067264804358, 'train_f1': 0.9736274038422952, 'test_loss': 3.5309142705165977, 'test_acc': 0.23799898938858008, 'test_p': 0.241696470341242, 'test_r': 0.23663377572293878, 'test_f1': 0.23325427760022696, 'bad_train_gifs': 0, 'bad_test_gifs': 0, 'train_chunks': 7910, 'test_chunks': 1979}
 Early stopping: no improvement for 5 epochs.
Saved history to phase2_strict_k6_history.csv


In [14]:

# =========================
# 12) EVAL – STRICT: chunk-level + GIF-level aggregation
# =========================
from collections import defaultdict

ckpt = torch.load(CKPT_STRICT, map_location=device)
model = init_model()
model.load_state_dict(ckpt["model"])
model.eval()
print("Loaded:", CKPT_STRICT, "epoch:", ckpt["epoch"])
print("Snapshot:", ckpt["metrics"])

softmax = torch.nn.Softmax(dim=1)

# chunk-level
all_true, all_pred = [], []
with torch.no_grad():
    for x, y, gid in tqdm(test_loader):
        x = x.to(device)
        pred = model(x).argmax(1).cpu().numpy().tolist()
        all_pred.extend(pred); all_true.extend(y.numpy().tolist())

print("\n=== STRICT: Chunk-level ===")
print("Accuracy:", accuracy_score(all_true, all_pred))

# GIF-level aggregation (avg prob across chunks of the same GIF)
gif_probs = defaultdict(list)
gif_true  = {}

with torch.no_grad():
    for x, y, gid in tqdm(test_loader):
        x = x.to(device)
        probs = softmax(model(x)).cpu().numpy()
        y_np = y.numpy()
        for i in range(len(gid)):
            gif_probs[gid[i]].append(probs[i])
            gif_true[gid[i]] = int(y_np[i])

gif_y_true, gif_y_pred = [], []
for g, plist in gif_probs.items():
    avgp = np.mean(np.stack(plist, axis=0), axis=0)
    gif_y_true.append(gif_true[g])
    gif_y_pred.append(int(np.argmax(avgp)))

print("\n=== STRICT: GIF-level (avg softmax over chunks) ===")
print("Accuracy:", accuracy_score(gif_y_true, gif_y_pred))
print(classification_report(gif_y_true, gif_y_pred, target_names=Z_CLASSES, digits=4, zero_division=0))
print("Confusion Matrix:\n", confusion_matrix(gif_y_true, gif_y_pred))


Loaded: phase2_strict_k6_best.pt epoch: 1
Snapshot: {'epoch': 1, 'train_loss': 1.3247896808792243, 'train_acc': 0.5093552465233881, 'train_p': 0.5088150491117062, 'train_r': 0.508263642505451, 'train_f1': 0.5065730054814881, 'test_loss': 2.3176620476173633, 'test_acc': 0.2627589691763517, 'test_p': 0.28252310749697623, 'test_r': 0.26368015658271937, 'test_f1': 0.2612885817364744, 'bad_train_gifs': 0, 'bad_test_gifs': 0, 'train_chunks': 7910, 'test_chunks': 1979}


  0%|          | 0/66 [00:00<?, ?it/s]


=== STRICT: Chunk-level ===
Accuracy: 0.2627589691763517


  0%|          | 0/66 [00:00<?, ?it/s]


=== STRICT: GIF-level (avg softmax over chunks) ===
Accuracy: 0.26666666666666666
              precision    recall  f1-score   support

   happiness     0.2267    0.2833    0.2519        60
satisfaction     0.2778    0.2500    0.2632        60
    surprise     0.3043    0.1167    0.1687        60
     sadness     0.4255    0.3333    0.3738        60
       anger     0.2292    0.1833    0.2037        60
     disgust     0.1948    0.2500    0.2190        60
        fear     0.2812    0.4500    0.3462        60

    accuracy                         0.2667       420
   macro avg     0.2771    0.2667    0.2609       420
weighted avg     0.2771    0.2667    0.2609       420

Confusion Matrix:
 [[17  5  4  0  6 15 13]
 [10 15  3  5  9  9  9]
 [11  6  7  3  6  8 19]
 [ 9  8  2 20  7  8  6]
 [ 7  8  1  5 11 15 13]
 [ 8  8  2  9  9 15  9]
 [13  4  4  5  0  7 27]]
